In [ ]:
# read the final agreed labels
import pandas as pd
labeled_prs = pd.read_csv('annotation/final_dataset_with_agreed_labels.csv')
print(len(labeled_prs))
labeled_prs = labeled_prs.dropna(subset=['sub-category'])
print(len(labeled_prs))

Labels mapping

In [ ]:

taxonomy = {
    "Dependency management": [
        "Version consistency",
        "Action version update",
        "Environment tooling update",
        "Dependencies expansion",
        "Dependency configuration",
    ],
    "Workflow configuration": [
        "Conditional execution configuration",
        "Test configuration",
        "Build environment configuration",
        "Matrix coverage expansion",
        "Runner configuration",
        "Artifact configuration",
    ],
    "Error Fixing": [
        "Package manager fix",
        "Dependency config fix",
        "Syntax error",
        "Test code fix",
        "Fix step input",
        "Cache config fix",
    ],
    "Workflow refactoring": [
        "Redundant task removal",
        "Task decomposition",
        "Task composition",
        "Action refactoring",
        "Renaming",
    ],
    "CI Task addition": [
        "Test task addition",
        "Lint task addition",
        "Publish & release task addition",
        "Build task addition",
    ],
    "Security & restrictions": [
        "Security scan addition",
        "Resources access restriction",
        "Artifact integrity validation",
    ],
    "Performance": [
        "Build speed optimization",
        "Build waste optimization",
        "Caching implementation",
    ],
    "Project Governance": [
        "Workflow reporting",
        "Pull request governance",
        "Release notes enrichment",
        "Documentation addition",
    ],
    "Agentic AI integration": [
        "AI test integration",
        "AI repository governance",
        "AI review integration",
    ],
    "Migration": [
        "Runtime migration",
        "Platform migration",
        "Action migration",
        "Tool migration",
    ],
}

In [ ]:
# Normalize taxonomy to lower case
taxonomy_lc = {
    main_category.lower(): [sub_category.lower() for sub_category in sub_categories]
    for main_category, sub_categories in taxonomy.items()
}


# Build sub-category -> main category mapping
sub_to_taxonomy = {
    sub_category: main_category
    for main_category, sub_categories in taxonomy_lc.items()
    for sub_category in sub_categories
}

# Normalize the dataset sub-category labels
labeled_prs['sub-category'] = labeled_prs['sub-category'].astype(str).str.strip().str.lower()

# Create the main taxonomy category column
labeled_prs['category'] = labeled_prs['sub-category'].map(sub_to_taxonomy)

# Check for labels that were not mapped
unmapped = labeled_prs.loc[labeled_prs['category'].isna(), 'sub-category'].unique()
print('Unmapped sub-categories:', unmapped)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

PLOT_DIR = Path('../figures/rq4')
PLOT_DIR.mkdir(exist_ok=True)

PR_TYPE_ORDER = ['ai', 'human']
PR_TYPE_COLORS = {'ai': '#1f77b4', 'human': '#ff7f0e'}

def title_label(value):
    return str(value).title().replace('Ai ', 'AI ').replace('Ci ', 'CI ').replace('Pr ', 'PR ')

def label_bars(ax, fmt='{:.0f}', padding=3, fontsize=9):
    for container in ax.containers:
        labels = []
        for bar in container:
            height = bar.get_height()
            labels.append('' if height == 0 else fmt.format(height))
        ax.bar_label(container, labels=labels, padding=padding, fontsize=fontsize)

def style_axes(ax, grid_axis='y'):
    ax.spines['top'].set_color('#333333')
    ax.spines['right'].set_color('#333333')
    ax.spines['left'].set_color('#333333')
    ax.spines['bottom'].set_color('#333333')
    if grid_axis:
        ax.grid(axis=grid_axis, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)


In [ ]:
# High-level labels by PR type: percentage within each PR type
high_level_by_pr_type = (
    labeled_prs
    .groupby(['category', 'pr_type'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=PR_TYPE_ORDER, fill_value=0)
)
high_level_by_pr_type = high_level_by_pr_type.loc[
    high_level_by_pr_type.sum(axis=1).sort_values(ascending=False).index
]
high_level_by_pr_type_percent = high_level_by_pr_type.div(high_level_by_pr_type.sum(axis=0), axis=1) * 100
high_level_by_pr_type.index = high_level_by_pr_type.index.map(title_label)
high_level_by_pr_type_percent.index = high_level_by_pr_type.index

fig, ax = plt.subplots(figsize=(14, 6))
high_level_by_pr_type_percent.plot(
    kind='bar',
    ax=ax,
    color=[PR_TYPE_COLORS['ai'], PR_TYPE_COLORS['human']],
    width=0.82,
)
ax.set_title('High-level labels by PR type — percentage', fontsize=16)
ax.set_ylabel('Percentage of PRs (%)')
ax.set_xlabel('')
ax.legend(title='PR type')
ax.tick_params(axis='x', rotation=35)
for label in ax.get_xticklabels():
    label.set_ha('right')
label_bars(ax, fmt='{:.1f}%', padding=3, fontsize=9)
style_axes(ax)
ax.set_ylim(0, high_level_by_pr_type_percent.to_numpy().max() * 1.18)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'plot_10_high_level_labels_by_pr_type_counts_with_values.png', dpi=300, bbox_inches='tight')
plt.show()

